# BuildFlowMatch (waft-a2, dav2) - demo minimal: checkpoint + 1 pas fine-tuning + inferenta

Singurul lucru pe care il aduci tu este checkpoint-ul `.pth`. Restul e deja inclus in repo:
o pereche de imagini exemplu cu flux optic cunoscut (`sample_data/`), scriptul de
fine-tuning (1 pas) si vizualizarea inainte/dupa.

## 1. Clonare repo
Inlocuieste `REPO_URL` cu URL-ul repo-ului tau, daca ai facut fork.

In [ ]:
REPO_URL = "https://github.com/rosemariestoica13/BuildFlowMatch.git"

!git clone $REPO_URL buildflowmatch
%cd buildflowmatch

## 2. Instalare dependinte

In [ ]:
!pip install -q -r requirements.txt

## 3. Checkpoint BuildFlowMatch de pornire
Incarca fisierul `.pth` (checkpoint-ul waft-a2/dav2, ex. cel din Google Drive-ul proiectului original) direct din calculator. Acesta e singurul upload necesar pentru date/model.

In [ ]:
import os
os.makedirs('checkpoints', exist_ok=True)

from google.colab import files
uploaded = files.upload()  # selecteaza checkpoint-ul .pth
for fname in uploaded:
    os.rename(fname, os.path.join('checkpoints', 'sintel-gm-final.pth'))
print(os.listdir('checkpoints'))

## 4. Greutatile encoder-ului dav2 (Depth Anything V2, vits)
Incearca intai descarcarea automata de pe Hugging Face. Daca nu merge (retea/permisiuni), incarca manual fisierul cu celula de mai jos.

In [ ]:
import os
os.makedirs('depth-anything-ckpts', exist_ok=True)
url = "https://huggingface.co/depth-anything/Depth-Anything-V2-Small/resolve/main/depth_anything_v2_vits.pth"
dst = 'depth-anything-ckpts/depth_anything_v2_vits.pth'
!wget -q --show-progress -O $dst $url
print(os.path.getsize(dst) if os.path.exists(dst) else 'download failed - incarca manual mai jos')

In [ ]:
# Ruleaza doar daca descarcarea automata de mai sus a esuat
from google.colab import files
uploaded = files.upload()  # selecteaza depth_anything_v2_vits.pth
for fname in uploaded:
    os.rename(fname, 'depth-anything-ckpts/depth_anything_v2_vits.pth')

## 5. Fine-tuning (1 pas) + inferenta
Foloseste perechea de imagini exemplu inclusa in repo, `sample_data/` (flux optic
cunoscut, generat sintetic) - nu e nevoie sa incarci propriul set de date.

In [ ]:
!python finetune_demo.py \
    --cfg config/a2/dav2/sintel-gm.json \
    --ckpt checkpoints/sintel-gm-final.pth \
    --data_dir sample_data \
    --steps 1 \
    --out_ckpt checkpoints/finetuned.pth

## 6. Vizualizare inainte / dupa fine-tuning

In [ ]:
import matplotlib.pyplot as plt
import cv2

before = cv2.cvtColor(cv2.imread('demo_out/flow_before.jpg'), cv2.COLOR_BGR2RGB)
after = cv2.cvtColor(cv2.imread('demo_out/flow_after.jpg'), cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(before); ax[0].set_title('Flux optic - inainte de fine-tuning'); ax[0].axis('off')
ax[1].imshow(after); ax[1].set_title('Flux optic - dupa fine-tuning'); ax[1].axis('off')
plt.show()